# Day 4 — Local MySQL: Install, Load CSV, Verify

**Objective:** Install or start a local MySQL server (native or Docker), connect from Python using SQLAlchemy, load `sales_clean.csv` into a `sales` table, and run basic verification queries. This notebook includes safe Python code for the load step and short OS-specific install hints.

## 1) Quick install hints (choose your OS)

macOS (Homebrew):
  1. `brew install mysql`
  2. `brew services start mysql`
  3. Run `mysql -u root` and create user/db:
     `CREATE USER 'apprentice'@'localhost' IDENTIFIED BY 'apprentice_pw';`
     `CREATE DATABASE apx_bakery;`
     `GRANT ALL PRIVILEGES ON apx_bakery.* TO 'apprentice'@'localhost';`

Ubuntu/Debian:
  1. `sudo apt update && sudo apt install mysql-server`
  2. `sudo systemctl start mysql`
  3. Run `sudo mysql_secure_installation` then `sudo mysql -u root` and create user/db as above.

Windows:
  Use the MySQL Installer from dev.mysql.com and follow GUI steps. After install open MySQL Shell and create the apprentice user and database as shown above.

Docker fallback (fast and reproducible):
  ```bash
  docker run --name mysql-apprentice -e MYSQL_ROOT_PASSWORD=apprentice_pw -e MYSQL_DATABASE=apx_bakery -e MYSQL_USER=apprentice -e MYSQL_PASSWORD=apprentice_pw -p 3306:3306 -d mysql:8.0
  ```
  Wait ~10-20s for the server to start. Check `docker logs mysql-apprentice` if needed.

## 2) Python: connect, load CSV into `sales`, and run basic checks

Make sure you have the Python packages installed: `pip install pandas sqlalchemy pymysql`

The code cell below reads `sales_clean.csv` (creates a tiny sample if missing), connects to MySQL using environment variables (with sensible defaults), writes the table using `if_exists='replace'` for idempotence, and performs verification queries.

In [ ]:
import os
import time
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text

# Configuration (use env vars to override defaults)
DB_USER = os.environ.get('DB_USER', 'apprentice')
DB_PASS = os.environ.get('DB_PASS', 'apprentice_pw')
DB_HOST = os.environ.get('DB_HOST', 'localhost')
DB_PORT = int(os.environ.get('DB_PORT', 3306))
DB_NAME = os.environ.get('DB_NAME', 'apx_bakery')

conn_str = f'mysql+pymysql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
print('Connection string:', conn_str)

# Ensure a cleaned CSV exists; create a tiny sample if missing
csv_path = Path('sales_clean.csv')
if not csv_path.exists():
    sample = [
        {'scraped_at': '2026-06-30T12:00:00', 'item': 'Croissant', 'price': 2.5, 'quantity': 1, 'total': 2.5, 'store': 'Main', 'url': 'u1'},
        {'scraped_at': '2026-06-30T12:05:00', 'item': 'Baguette', 'price': 3.0, 'quantity': 2, 'total': 6.0, 'store': 'Main', 'url': 'u2'},
        {'scraped_at': '2026-07-01T09:00:00', 'item': 'Croissant', 'price': 2.5, 'quantity': 3, 'total': 7.5, 'store': 'Branch', 'url': 'u3'},
    ]
    pd.DataFrame(sample).to_csv(csv_path, index=False, encoding='utf-8')
    print('Wrote sample sales_clean.csv — replace with your real file')

# Read CSV
try:
    df = pd.read_csv(csv_path, parse_dates=['scraped_at'], encoding='utf-8')
    print('Loaded rows:', len(df))
except Exception as e:
    print('Failed to read CSV:', e)
    raise

# Create SQLAlchemy engine and test connection with retries
engine = create_engine(conn_str, pool_pre_ping=True)
last_exc = None
for attempt in range(5):
    try:
        with engine.connect() as conn:
            conn.execute(text('SELECT 1'))
        print('Successfully connected to the DB')
        last_exc = None
        break
    except Exception as e:
        last_exc = e
        print(f'Connection attempt {attempt+1} failed:', e)
        time.sleep(2)

if last_exc is not None:
    raise last_exc

# Write to DB using replace (idempotent for learning)
try:
    df.to_sql('sales', engine, index=False, if_exists='replace', chunksize=500)
    print('Wrote', len(df), 'rows to sales table')
except Exception as e:
    print('Failed to write to DB:', e)
    raise

# Verification queries
try:
    q_count = pd.read_sql('SELECT COUNT(*) AS cnt FROM sales', engine)
    print('Row count in sales table:', int(q_count['cnt'].iloc[0]))

    q_top = pd.read_sql('SELECT item, SUM(total) AS revenue FROM sales GROUP BY item ORDER BY revenue DESC LIMIT 5', engine)
    print('Top items from DB:')
    print(q_top.to_string(index=False))

    q_daily = pd.read_sql('SELECT DATE(scraped_at) AS d, SUM(total) AS revenue FROM sales GROUP BY d ORDER BY d DESC LIMIT 10', engine)
    print('Recent daily totals:')
    print(q_daily.to_string(index=False))
except Exception as e:
    print('Verification queries failed:', e)
    raise
